# 08. Illumination Correction

Các phương pháp:

1. CLAHE
2. Gamma correction
3. White balance

Không áp dụng tất cả phương pháp cho mọi ảnh mặc định.

Mỗi phương pháp tạo một preprocessing branch để đánh giá downstream retrieval performance.

Các metric chính:

- Recall@1
- Recall@10
- Recall@100
- mAP

Image-level metrics:

- brightness statistics
- contrast
- saturation
- color temperature proxy

## 1. CLAHE

In [ ]:
def clahe_rgb(image,clip_limit=2.0,tile_grid=(8, 8)):

    lab = cv2.cvtColor(image,cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=tile_grid
    )

    l = clahe.apply(l)
    result = cv2.merge([l,a,b])

    return cv2.cvtColor(result, cv2.COLOR_LAB2RGB)

## 2. Gamma corection

In [ ]:
def gamma_correction(image,gamma):
    inv_gamma = 1.0 / gamma
    table = np.array([
        (
            (i / 255.0)
            ** inv_gamma
        ) * 255
        for i in range(256)
    ]).astype(
        np.uint8
    )
    return cv2.LUT(
        image,
        table
    )

In [ ]:
gammas = [
    0.7,
    0.85,
    1.0,
    1.15,
    1.3
]

## 3. White balance

In [ ]:
def gray_world_white_balance(image):
    image = image.astype(
        np.float32
    )
    mean_rgb = image.mean(
        axis=(0, 1)
    )
    mean_gray = mean_rgb.mean()

    scale = (
        mean_gray
        /
        (mean_rgb + 1e-6)
    )

    balanced = (
        image * scale
    )

    return np.clip(
        balanced,
        0,
        255
    ).astype(
        np.uint8
    )

## 4. Visualization

In [ ]:
sample = df.sample(5,random_state=42)

fig, axes = plt.subplots(5,4,figsize=(12, 15))

for i, (_, row) in enumerate(sample.iterrows()):
    path = (
        DATASET_ROOT
        /
        row["path"]
    )

    image = Image.open(path)
    image = ImageOps.exif_transpose(image)
    image = np.array(image.convert("RGB"))

    clahe = clahe_rgb(image)
    gamma = gamma_correction(image,gamma=0.7)

    wb = gray_world_white_balance(image)

    images = [
        image,
        clahe,
        gamma,
        wb
    ]

    titles = [
        "Original",
        "CLAHE",
        "Gamma",
        "White Balance"
    ]

    for j, (
        img,
        title
    ) in enumerate(
        zip(images, titles)
    ):

        axes[i, j].imshow(img)
        axes[i, j].set_title(title)
        axes[i, j].axis("off")

plt.tight_layout()

## Image-level illumination statistics

In [ ]:
def illumination_statistics(image):

    gray = cv2.cvtColor(image,cv2.COLOR_RGB2GRAY)

    return {
        "mean_brightness": float(
            gray.mean()
        ),
        "std_brightness": float(
            gray.std()
        ),
        "min_brightness": float(
            gray.min()
        ),
        "max_brightness": float(
            gray.max()
        )
    }

In [ ]:
original_stats = []
clahe_stats = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    path = DATASET_ROOT / row["path"]
    image = Image.open(path)
    image = ImageOps.exif_transpose(image)
    image = np.array(image.convert("RGB"))

    processed = clahe_rgb(image)

    s1 = illumination_statistics(image)
    s2 = illumination_statistics(processed)

    original_stats.append(s1)
    clahe_stats.append(s2)